# 1 - Imports

In [3]:
%reload_ext autoreload
%autoreload 2

In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

In [5]:
import pandas as pd
import numpy as np

In [6]:
from src.utils import config, io
from src.features import selection, pruning

/Users/hippolytegrandet/Desktop/Dev/country_risk_rating/.venv/lib/python3.9/site-packages/pypdf/_crypt_providers/_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


# 2 - Feature Selection (Independent of Target)

In [7]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'merged_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,GE.EST,RQ.EST,IC.BRE.BI.OS,IC.BRE.BE.OS,IQ.CPA.PADM.XQ,FS.AST.PRVT.GD.ZS,FM.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,NaN,108.686031,46.609,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,NaN,109.586048,46.562,-2.173946,-2.080253,NaN,NaN,NaN,NaN,NaN
AFG-2001,2001,AFG,-,MEA,MNA,IDX,LIC,2.813572e+09,-9.431974,1.516633e+10,...,NaN,110.219341,46.526,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AFG-2002,2002,AFG,-,MEA,MNA,IDX,LIC,3.825701e+09,28.600001,1.980700e+10,...,NaN,110.534611,46.505,-1.587687,-1.811546,NaN,NaN,NaN,NaN,NaN
AFG-2003,2003,AFG,-,MEA,MNA,IDX,LIC,4.520947e+09,8.832278,2.198200e+10,...,NaN,110.557540,46.497,-1.175768,-1.463108,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,3.335770e+10,-6.332450,6.361373e+10,...,75.001228,85.447906,65.795,-1.310435,-1.486515,NaN,NaN,3.0,3.428022,3.428022
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,3.198033e+10,-7.816951,6.488043e+10,...,NaN,84.384381,64.665,-1.342368,-1.434415,NaN,NaN,3.0,3.642132,3.642132
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,4.128767e+10,8.468017,7.625453e+10,...,NaN,83.384953,65.397,-1.290561,-1.386109,NaN,NaN,3.0,4.759522,4.759522


In [8]:
selection_params = {
    'max_missing_ratio': 0.6,
    'low_var_threshold': 0.001,
    'max_corr': 0.95,
    'top_k_mi': 100
}

In [9]:
dataset = selection.get_dataset_feature_selected(
    dataset, 
    selection_params, 
    verbose=True
)

Initial Dataset Shape: (5025, 85)
Drop Null Target Shape: (4204, 85)
Filter Missingness Shape: (4204, 79)
Filter Low Variance Shape: (4204, 78)
Filter Correlated Shape: (4204, 64)
Final Dataset Shape: (4204, 64)


In [10]:
io.save_csv(dataset, config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index=True)

# 3 - Feature Pruning (Aligned with Target)

In [11]:
dataset = io.load_csv(config.INTERIM_DATA_DIR / 'feature_selected_dataset.csv', index_col=0)
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SP.POP.TOTL,SP.POP.GROW,SP.URB.TOTL.IN.ZS,EN.POP.DNST,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,GE.EST,RQ.EST,FS.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-1999,1999,AFG,7,MEA,MNA,IDX,LIC,NaN,NaN,NaN,...,19887785.0,3.728116,18.390327,30.491981,NaN,108.686031,46.609,NaN,NaN,NaN
AFG-2000,2000,AFG,7,MEA,MNA,IDX,LIC,3.521418e+09,NaN,1.637703e+10,...,20130327.0,1.212176,18.558200,30.863847,NaN,109.586048,46.562,-2.173946,-2.080253,NaN
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,1.010930e+10,3.924984,3.532112e+10,...,26482622.0,2.186546,21.124148,40.603195,NaN,106.334376,46.682,-1.527795,-1.607167,9.388328
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,1.241615e+10,21.390528,4.314095e+10,...,27466101.0,3.646381,21.688548,42.111067,NaN,104.799467,46.746,-1.507752,-1.664508,10.584131
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,1.585667e+10,14.362441,4.993663e+10,...,28284089.0,2.934687,22.261480,43.365207,NaN,103.144456,46.815,-1.478316,-1.516528,11.575044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,3.335770e+10,-6.332450,6.361373e+10,...,15271368.0,1.563534,36.183764,39.476200,75.001228,85.447906,65.795,-1.310435,-1.486515,3.428022
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,3.198033e+10,-7.816951,6.488043e+10,...,15526888.0,1.659353,37.009767,40.136714,NaN,84.384381,64.665,-1.342368,-1.434415,3.642132
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,4.128767e+10,8.468017,7.625453e+10,...,15797210.0,1.726011,37.869206,40.835492,NaN,83.384953,65.397,-1.290561,-1.386109,4.759522


In [12]:
dataset = dataset[(dataset.isna().sum(axis=1) / len(dataset.columns)) <= 0.5]
dataset

,YEAR,ISO3_COUNTRY_CODE,OECD_RATING,GEO_REGION,ADMIN_REGION,LENDING_TYPE,INCOME_GROUP,NY.GDP.MKTP.CD,NY.GDP.MKTP.KD.ZG,NY.GDP.MKTP.PP.CD,...,SP.POP.TOTL,SP.POP.GROW,SP.URB.TOTL.IN.ZS,EN.POP.DNST,SE.PRM.CUAT.ZS,SP.POP.DPND,SL.TLF.CACT.ZS,GE.EST,RQ.EST,FS.AST.PRVT.GD.ZS
COUNTRY_PERIOD_INDEX,,,,,,,,,,,,,,,,,,,,,
AFG-2008,2008,AFG,7,MEA,MNA,IDX,LIC,1.010930e+10,3.924984,3.532112e+10,...,26482622.0,2.186546,21.124148,40.603195,NaN,106.334376,46.682,-1.527795,-1.607167,9.388328
AFG-2009,2009,AFG,7,MEA,MNA,IDX,LIC,1.241615e+10,21.390528,4.314095e+10,...,27466101.0,3.646381,21.688548,42.111067,NaN,104.799467,46.746,-1.507752,-1.664508,10.584131
AFG-2010,2010,AFG,7,MEA,MNA,IDX,LIC,1.585667e+10,14.362441,4.993663e+10,...,28284089.0,2.934687,22.261480,43.365207,NaN,103.144456,46.815,-1.478316,-1.516528,11.575044
AFG-2011,2011,AFG,7,MEA,MNA,IDX,LIC,1.780510e+10,0.426355,5.118418e+10,...,29347708.0,3.691503,22.820569,44.995949,NaN,101.458308,46.884,-1.474100,-1.536324,4.959180
AFG-2012,2012,AFG,7,MEA,MNA,IDX,LIC,1.990733e+10,12.752287,6.076647e+10,...,30560034.0,4.047863,23.343439,46.854689,NaN,99.773131,46.956,-1.375535,-1.192580,4.472104
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZWE-2019,2019,ZWE,7,SSF,SSA,IDB,LMC,3.335770e+10,-6.332450,6.361373e+10,...,15271368.0,1.563534,36.183764,39.476200,75.001228,85.447906,65.795,-1.310435,-1.486515,3.428022
ZWE-2020,2020,ZWE,7,SSF,SSA,IDB,LMC,3.198033e+10,-7.816951,6.488043e+10,...,15526888.0,1.659353,37.009767,40.136714,NaN,84.384381,64.665,-1.342368,-1.434415,3.642132
ZWE-2021,2021,ZWE,7,SSF,SSA,IDB,LMC,4.128767e+10,8.468017,7.625453e+10,...,15797210.0,1.726011,37.869206,40.835492,NaN,83.384953,65.397,-1.290561,-1.386109,4.759522


In [13]:
X = dataset.drop(columns=['OECD_RATING']) # Drop Target
X = X.drop(columns=['ISO3_COUNTRY_CODE']) # Drop Country ID (maybe YEAR)
y = dataset['OECD_RATING']

In [14]:
io.save_csv(X, config.PROCESSED_DATA_DIR / 'X.csv', index=True)

In [15]:
io.save_csv(y, config.PROCESSED_DATA_DIR / 'y.csv', index=True)